# Full Quran + Arabic Tafsir Chroma Builder

This notebook is the full standalone **data pipeline before retrieval**.

It does everything inline inside Kaggle:
- fetches the Quran and tafsir payloads from Quran Hub
- validates the API schema and cached files
- normalizes Arabic text for dense retrieval and BM25
- builds ayah-level chunks with rich metadata
- embeds passages with `jinaai/jina-embeddings-v3`
- ingests the vectors into persistent ChromaDB
- exports the Chroma directory as `quran_full.zip`
- runs a large Arabic-only validation suite with edge cases

The notebook is intentionally Arabic-only:
- Quran: `quran-uthmani`
- Tafsir: `ar.tabari`
- Tafsir: `ar.muyassar`
- Tafsir: `ar.mukhtasar`


## Pipeline Design

The notebook is split into explicit stages so the ingestion flow is maintainable and production-oriented.

Stages:
1. Configuration
2. Data fetching with retry and cache safety
3. Arabic normalization (safety-first)
4. Chunk construction and metadata generation
5. Embedding generation
6. Chroma ingestion
7. Corpus integrity validation
8. Hybrid retrieval validation

Normalization policy (Qur'anic safety constraints):
- canonical text is preserved verbatim in `documents`
- retrieval forms are separated into derived metadata fields
- BM25 document normalization is conservative
- query normalization is cautious and explicit
- risky folds are disabled by default: `ة↔ه`, `ى↔ي`, aggressive Uthmani rewriting

Chunking policy:
- `1 ayah = 1 chunk`
- Quran and tafsir remain separate content types
- each chunk stores metadata for filtering, tracing, citation, and future source expansion

Retrieval policy:
- exact ayah-reference lookup fallback (e.g., `2:255`, `٢:٢٥٥`)
- hybrid dense + BM25 with reciprocal rank fusion
- explicit `content_type` and `edition_identifier` filtering

Validation policy:
- strict ayah-count and reference alignment checks
- deterministic chunk-ID checks
- metadata completeness and canonical hash checks
- retrieval probes for exact, near-exact, and filtered search

In [ ]:
%pip install -q chromadb rank-bm25 httpx einops safetensors
%pip install -q "huggingface-hub>=0.34.0,<1.0" "transformers==4.57.1" "sentence-transformers==5.1.2"

## Configuration

Adjust only this cell if you want different runtime behavior.

Recommended Kaggle setup:
- GPU runtime for faster embedding
- keep `refresh_api_cache=False` after the first successful run
- reduce `embedding_batch_size` if you hit memory pressure


In [1]:
"""Production-quality standalone Quran + tafsir ingestion pipeline (accuracy-first).

Safety policy for Qur'anic retrieval:
- Canonical ayah text is preserved verbatim in storage/document fields.
- Retrieval normalization is separated into derived fields only.
- High-risk Arabic folding rules are not applied destructively.
- Query-time/document-time lexical aids are conservative and explicit.
"""

from __future__ import annotations

import gc
import hashlib
import json
import logging
import os
import re
import shutil
import time
import unicodedata
from collections import Counter
from dataclasses import dataclass, field
from datetime import datetime, timezone
from pathlib import Path
from tempfile import NamedTemporaryFile
from typing import Any, Iterator, Sequence

os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")

import chromadb
import httpx
import numpy as np
import torch
from rank_bm25 import BM25Okapi
from sentence_transformers import SentenceTransformer

LOGGER = logging.getLogger(__name__)
EXPECTED_QURAN_AYAH_TOTAL = 6236
EXPECTED_QURAN_SURAH_TOTAL = 114
PIPELINE_VERSION = "2.2.0-quran-safety"
CHUNK_SCHEMA_VERSION = "2.2"


@dataclass(frozen=True, slots=True)
class EditionSpec:
    """Edition configuration for one Quran Hub text corpus."""

    identifier: str
    kind: str
    language: str = "ar"
    tafsir_author: str | None = None
    expected_ayah_count: int = EXPECTED_QURAN_AYAH_TOTAL

    @property
    def content_type(self) -> str:
        return "tafsir" if self.kind == "tafsir" else "quran_ayah"


DEFAULT_EDITIONS: tuple[EditionSpec, ...] = (
    EditionSpec("quran-uthmani", kind="quran"),
    EditionSpec("ar.tabari", kind="tafsir", tafsir_author="الطبري"),
    EditionSpec("ar.muyassar", kind="tafsir", tafsir_author="التفسير الميسر"),
    EditionSpec("ar.mukhtasar", kind="tafsir", tafsir_author="المختصر في التفسير"),
)


@dataclass(slots=True)
class PipelineConfig:
    """Runtime configuration for the standalone ingestion pipeline."""

    workspace_dir: Path = Path("/kaggle/working/quran_rag_build")
    api_base_url: str = "https://api.quranhub.com/v1"
    embedding_model_name: str = "jinaai/jina-embeddings-v3"
    embedding_fallback_model_name: str = "intfloat/multilingual-e5-large"
    embedding_dimension: int = 1024
    embedding_batch_size: int = 2
    embedding_min_batch_size: int = 1
    embedding_max_seq_length: int = 300
    embedding_device: str = "cuda"
    embedding_auto_cpu_max_gpu_memory_gb: float = 20.0
    embedding_query_task: str = "retrieval.query"
    embedding_passage_task: str = "retrieval.passage"
    embedding_query_prefix: str = "query: "
    embedding_passage_prefix: str = "passage: "
    chroma_collection_name: str = "quran_tafsir_ar"
    chroma_directory_name: str = "quran_full"
    export_zip_name: str = "quran_full.zip"
    request_timeout_seconds: float = 180.0
    request_max_retries: int = 4
    request_backoff_seconds: float = 2.0
    refresh_api_cache: bool = False
    reset_chroma_directory: bool = True
    chroma_upsert_batch_size: int = 128
    edition_specs: tuple[EditionSpec, ...] = field(default_factory=lambda: DEFAULT_EDITIONS)

    @property
    def raw_cache_dir(self) -> Path:
        return self.workspace_dir / "raw_cache"

    @property
    def chroma_dir(self) -> Path:
        return self.workspace_dir / self.chroma_directory_name

    @property
    def manifest_path(self) -> Path:
        return self.workspace_dir / "build_manifest.json"

    @property
    def export_zip_path(self) -> Path:
        return self.workspace_dir / self.export_zip_name


@dataclass(slots=True)
class ChunkMetadata:
    """Metadata stored alongside each ayah chunk."""

    content_type: str
    source: str
    source_family: str
    edition_identifier: str
    edition_name: str
    language: str
    surah_number: int
    ayah_number_in_surah: int
    ayah_number_global: int
    ayah_ref: str
    surah_name_arabic: str
    surah_name_english: str | None
    juz: int | None
    manzil: int | None
    page: int | None
    ruku: int | None
    hizb_quarter: int | None
    sajda: bool
    revelation_type: str | None
    tafsir_author: str | None
    raw_source_text: str
    canonical_text: str
    bm25_normalized_text: str
    embedding_normalized_text: str
    canonical_text_sha256: str
    normalization_profile: str
    chunk_schema_version: str
    pipeline_version: str
    ingested_at_utc: str
    metadata_fingerprint: str
    source_url: str

    def to_chroma(self) -> dict[str, str | int | float | bool]:
        output: dict[str, str | int | float | bool] = {}
        for key in self.__slots__:
            value = getattr(self, key)
            if value is None:
                continue
            if isinstance(value, (str, int, float, bool)):
                output[key] = value
            else:
                output[key] = str(value)
        return output


@dataclass(slots=True)
class DocumentChunk:
    """Ayah-aligned unit stored in the vector database."""

    chunk_id: str
    text: str
    text_for_embedding: str
    metadata: ChunkMetadata


def batched(items: Sequence[DocumentChunk], batch_size: int) -> Iterator[Sequence[DocumentChunk]]:
    for start in range(0, len(items), batch_size):
        yield items[start : start + batch_size]


class ArabicTextNormalizer:
    """Arabic normalization tuned for Qur'anic safety and retrieval correctness.

    Important constraints:
    - Canonical text is never rewritten destructively.
    - Doc-side lexical normalization is conservative.
    - Risky orthographic conflations are avoided (e.g., ة↔ه, ى↔ي).
    - Query/document token expansion uses only cautious alef-family folding.
    """

    _ZERO_WIDTH = str.maketrans("", "", "\ufeff\u200b\u200c\u200d\u200e\u200f\u2066\u2067\u2068\u2069")
    _DIACRITICS_RE = re.compile("[\u0610-\u061A\u064B-\u065F\u0670\u06D6-\u06ED\uFE70-\uFE7F]")
    _WHITESPACE_RE = re.compile(r"\s+")
    _WORD_RE = re.compile(r"[\u0600-\u06FF0-9]+")
    _ALEF_FOLD_MAP = str.maketrans({"أ": "ا", "إ": "ا", "آ": "ا", "ٱ": "ا", "ٲ": "ا", "ٳ": "ا", "ٵ": "ا"})
    _ARABIC_INDIC_DIGITS = str.maketrans("٠١٢٣٤٥٦٧٨٩", "0123456789")

    def normalize_unicode(self, text: str) -> str:
        return unicodedata.normalize("NFKC", text).translate(self._ZERO_WIDTH)

    def remove_diacritics(self, text: str) -> str:
        return self._DIACRITICS_RE.sub("", text)

    @staticmethod
    def remove_tatweel(text: str) -> str:
        return text.replace("ـ", "")

    def _collapse_whitespace(self, text: str) -> str:
        return self._WHITESPACE_RE.sub(" ", text).strip()

    def normalize_for_embedding(self, text: str) -> str:
        normalized = self.normalize_unicode(text)
        normalized = self.remove_diacritics(normalized)
        normalized = self.remove_tatweel(normalized)
        normalized = normalized.translate(self._ALEF_FOLD_MAP)
        return self._collapse_whitespace(normalized)

    def normalize_for_bm25_document(self, text: str) -> str:
        normalized = self.normalize_unicode(text)
        normalized = self.remove_diacritics(normalized)
        normalized = self.remove_tatweel(normalized)
        return self._collapse_whitespace(normalized)

    def normalize_for_bm25_query(self, text: str) -> str:
        normalized = self.normalize_for_bm25_document(text)
        normalized = normalized.translate(self._ALEF_FOLD_MAP)
        return self._collapse_whitespace(normalized)

    def tokenize_document_for_bm25(self, text: str) -> list[str]:
        base_tokens = self._WORD_RE.findall(self.normalize_for_bm25_document(text))
        output: list[str] = []
        for token in base_tokens:
            if len(token) <= 1:
                continue
            output.append(token)
            folded = token.translate(self._ALEF_FOLD_MAP)
            if folded != token and len(folded) > 1:
                output.append(folded)
        return output

    def tokenize_query_for_bm25(self, text: str) -> list[str]:
        base_tokens = self._WORD_RE.findall(self.normalize_for_bm25_query(text))
        return [token for token in base_tokens if len(token) > 1]

    def normalize_ayah_ref(self, query: str) -> str | None:
        compact = self.normalize_unicode(query).translate(self._ARABIC_INDIC_DIGITS)
        compact = self._collapse_whitespace(compact)
        match = re.search(r"\b(\d{1,3})\s*[:\-\/]\s*(\d{1,3})\b", compact)
        if not match:
            return None
        return f"{int(match.group(1))}:{int(match.group(2))}"


class QuranHubClient:
    """Robust Quran Hub client with retries, validation, and atomic cache writes."""

    _RETRYABLE_STATUSES = {408, 429, 500, 502, 503, 504}

    def __init__(self, config: PipelineConfig):
        self.config = config
        self.config.workspace_dir.mkdir(parents=True, exist_ok=True)
        self.config.raw_cache_dir.mkdir(parents=True, exist_ok=True)

    def fetch_all(self) -> dict[str, dict[str, Any]]:
        return {spec.identifier: self.fetch_edition(spec) for spec in self.config.edition_specs}

    def fetch_edition(self, spec: EditionSpec) -> dict[str, Any]:
        safe_name = spec.identifier.replace(".", "_").replace("-", "_")
        cache_path = self.config.raw_cache_dir / f"{safe_name}.json"
        if cache_path.exists() and not self.config.refresh_api_cache:
            cached = self._read_cached_json(cache_path)
            if cached is not None:
                return self._validate_payload(cached, spec)

        payload = self._request_json(f"/quran/{spec.identifier}")
        validated = self._validate_payload(payload, spec)
        self._write_json_atomic(cache_path, validated)
        return validated

    def _request_json(self, path: str) -> dict[str, Any]:
        last_error: Exception | None = None
        for attempt in range(1, self.config.request_max_retries + 1):
            try:
                with httpx.Client(
                    base_url=self.config.api_base_url,
                    timeout=self.config.request_timeout_seconds,
                    headers={"Accept": "application/json", "User-Agent": "Mozilla/5.0"},
                ) as client:
                    response = client.get(path)
                    if response.status_code in self._RETRYABLE_STATUSES:
                        raise httpx.HTTPStatusError("retryable status", request=response.request, response=response)
                    response.raise_for_status()
                    payload = response.json()
                if payload.get("code") != 200 or "data" not in payload:
                    raise ValueError(f"Unexpected Quran Hub payload for {path}: {payload}")
                return payload["data"]
            except (httpx.RequestError, httpx.HTTPStatusError, json.JSONDecodeError, ValueError) as error:
                last_error = error
                if attempt == self.config.request_max_retries:
                    break
                sleep_seconds = self.config.request_backoff_seconds * (2 ** (attempt - 1))
                LOGGER.warning("Retrying %s after %s", path, error)
                time.sleep(sleep_seconds)
        raise RuntimeError(f"Failed to fetch {path} after retries") from last_error

    @staticmethod
    def _read_cached_json(path: Path) -> dict[str, Any] | None:
        try:
            return json.loads(path.read_text(encoding="utf-8"))
        except (OSError, json.JSONDecodeError):
            corrupt_path = path.with_suffix(path.suffix + ".corrupt")
            path.replace(corrupt_path)
            LOGGER.warning("Moved corrupt cache file to %s", corrupt_path)
            return None

    @staticmethod
    def _write_json_atomic(path: Path, payload: dict[str, Any]) -> None:
        with NamedTemporaryFile("w", encoding="utf-8", delete=False, dir=path.parent) as handle:
            json.dump(payload, handle, ensure_ascii=False)
            temp_path = Path(handle.name)
        temp_path.replace(path)

    @staticmethod
    def _validate_payload(payload: dict[str, Any], spec: EditionSpec) -> dict[str, Any]:
        surahs = payload.get("surahs")
        edition = payload.get("edition")
        if not isinstance(surahs, list) or not isinstance(edition, dict):
            raise ValueError(f"Edition {spec.identifier} missing surahs/edition")

        total_ayahs = 0
        seen_refs: set[str] = set()
        seen_global_numbers: set[int] = set()
        seen_surah_numbers: set[int] = set()

        for surah in surahs:
            if not isinstance(surah, dict):
                raise ValueError(f"Invalid surah object in {spec.identifier}")
            if "ayahs" not in surah or "number" not in surah or "name" not in surah:
                raise ValueError(f"Invalid surah schema in {spec.identifier}")

            surah_number = int(surah["number"])
            if surah_number in seen_surah_numbers:
                raise ValueError(f"Duplicate surah number in {spec.identifier}: {surah_number}")
            seen_surah_numbers.add(surah_number)

            ayahs = surah["ayahs"]
            if not isinstance(ayahs, list):
                raise ValueError(f"Invalid ayah list in {spec.identifier}")

            for idx, ayah in enumerate(ayahs, start=1):
                required = {"number", "numberInSurah", "text"}
                if not isinstance(ayah, dict) or not required.issubset(ayah):
                    missing = required - set(ayah) if isinstance(ayah, dict) else required
                    raise ValueError(f"Invalid ayah schema in {spec.identifier}: missing {missing}")

                number_in_surah = int(ayah["numberInSurah"])
                global_number = int(ayah["number"])
                if number_in_surah != idx:
                    raise ValueError(
                        f"Non-sequential ayah numbering in {spec.identifier}, surah {surah_number}: "
                        f"expected {idx}, got {number_in_surah}"
                    )

                ayah_ref = f"{surah_number}:{number_in_surah}"
                if ayah_ref in seen_refs:
                    raise ValueError(f"Duplicate ayah_ref in {spec.identifier}: {ayah_ref}")
                seen_refs.add(ayah_ref)

                if global_number in seen_global_numbers:
                    raise ValueError(f"Duplicate global ayah number in {spec.identifier}: {global_number}")
                seen_global_numbers.add(global_number)

            total_ayahs += len(ayahs)

        if spec.kind == "quran":
            if len(surahs) != EXPECTED_QURAN_SURAH_TOTAL:
                raise ValueError(f"Edition {spec.identifier} expected {EXPECTED_QURAN_SURAH_TOTAL} surahs, got {len(surahs)}")
            if seen_surah_numbers != set(range(1, EXPECTED_QURAN_SURAH_TOTAL + 1)):
                raise ValueError("Quran surah numbers are incomplete or malformed")

        if total_ayahs != spec.expected_ayah_count:
            raise ValueError(f"Edition {spec.identifier} expected {spec.expected_ayah_count} ayahs, got {total_ayahs}")
        return payload


class QuranChunkBuilder:
    """Turn Quran Hub payloads into typed ayah chunks while preserving canonical text."""

    def __init__(self, config: PipelineConfig, normalizer: ArabicTextNormalizer):
        self.config = config
        self.normalizer = normalizer

    @staticmethod
    def _build_metadata_fingerprint(
        edition_identifier: str,
        ayah_ref: str,
        canonical_text_sha256: str,
        source_url: str,
    ) -> str:
        payload = f"{edition_identifier}|{ayah_ref}|{canonical_text_sha256}|{source_url}"
        return hashlib.sha256(payload.encode("utf-8")).hexdigest()

    def build(self, payloads: dict[str, dict[str, Any]]) -> list[DocumentChunk]:
        spec_by_id = {spec.identifier: spec for spec in self.config.edition_specs}
        chunks: list[DocumentChunk] = []
        ingested_at_utc = datetime.now(timezone.utc).isoformat()

        for edition_identifier, payload in payloads.items():
            spec = spec_by_id[edition_identifier]
            edition_name = payload["edition"].get("name", edition_identifier)
            for surah in payload["surahs"]:
                surah_number = int(surah["number"])
                for ayah in surah["ayahs"]:
                    ayah_number_in_surah = int(ayah["numberInSurah"])
                    ayah_number_global = int(ayah["number"])
                    ayah_ref = f"{surah_number}:{ayah_number_in_surah}"

                    raw_source_text = str(ayah["text"])
                    canonical_text = raw_source_text
                    if canonical_text != raw_source_text:
                        raise ValueError(f"Canonical text drift detected for {ayah_ref}/{edition_identifier}")
                    bm25_norm = self.normalizer.normalize_for_bm25_document(canonical_text)
                    embedding_norm = self.normalizer.normalize_for_embedding(canonical_text)
                    canonical_hash = hashlib.sha256(canonical_text.encode("utf-8")).hexdigest()
                    source_url = f"{self.config.api_base_url}/ayah/{ayah_ref}/{edition_identifier}"
                    metadata_fingerprint = self._build_metadata_fingerprint(
                        edition_identifier,
                        ayah_ref,
                        canonical_hash,
                        source_url,
                    )

                    metadata = ChunkMetadata(
                        content_type=spec.content_type,
                        source="tafsir" if spec.kind == "tafsir" else "quran",
                        source_family="quran_hub",
                        edition_identifier=edition_identifier,
                        edition_name=edition_name,
                        language=spec.language,
                        surah_number=surah_number,
                        ayah_number_in_surah=ayah_number_in_surah,
                        ayah_number_global=ayah_number_global,
                        ayah_ref=ayah_ref,
                        surah_name_arabic=surah["name"],
                        surah_name_english=surah.get("englishName"),
                        juz=ayah.get("juz"),
                        manzil=ayah.get("manzil"),
                        page=ayah.get("page"),
                        ruku=ayah.get("ruku"),
                        hizb_quarter=ayah.get("hizbQuarter"),
                        sajda=bool(ayah.get("sajda")) if ayah.get("sajda") is not None else False,
                        revelation_type=surah.get("revelationType"),
                        tafsir_author=spec.tafsir_author,
                        raw_source_text=raw_source_text,
                        canonical_text=canonical_text,
                        bm25_normalized_text=bm25_norm,
                        embedding_normalized_text=embedding_norm,
                        canonical_text_sha256=canonical_hash,
                        normalization_profile="canonical-preserved|bm25-doc-conservative|dense-light",
                        chunk_schema_version=CHUNK_SCHEMA_VERSION,
                        pipeline_version=PIPELINE_VERSION,
                        ingested_at_utc=ingested_at_utc,
                        metadata_fingerprint=metadata_fingerprint,
                        source_url=source_url,
                    )

                    if spec.kind == "tafsir":
                        prefix = f"تفسير {edition_name} {ayah_ref}: "
                    else:
                        prefix = f"سورة {surah['name']} آية {ayah_number_in_surah}: "

                    chunks.append(
                        DocumentChunk(
                            chunk_id=self._build_chunk_id(spec, surah_number, ayah_number_in_surah),
                            text=canonical_text,
                            text_for_embedding=self.config.embedding_passage_prefix + prefix + embedding_norm,
                            metadata=metadata,
                        )
                    )
        return chunks

    @staticmethod
    def _build_chunk_id(spec: EditionSpec, surah_number: int, ayah_number_in_surah: int) -> str:
        if spec.kind == "quran" and spec.identifier == "quran-uthmani":
            return f"quran_ar_{surah_number}_{ayah_number_in_surah}"
        safe_edition = spec.identifier.replace(".", "_").replace("-", "_")
        return f"{spec.content_type}_{safe_edition}_ar_{surah_number}_{ayah_number_in_surah}"


class EmbeddingModel:
    """Thin wrapper around Jina v3 embeddings with strict output safety checks."""

    def __init__(self, config: PipelineConfig, normalizer: ArabicTextNormalizer):
        self.config = config
        self.normalizer = normalizer
        self._model: SentenceTransformer | None = None
        self._effective_embedding_device = self._resolve_embedding_device()
        self._force_cpu_for_embeddings = self._effective_embedding_device == "cpu"

    def _resolve_embedding_device(self) -> str:
        requested = self.config.embedding_device.lower()
        if requested == "cpu":
            return "cpu"
        if not torch.cuda.is_available():
            return "cpu"
        if requested == "cuda":
            return "cuda"
        try:
            total_gb = torch.cuda.get_device_properties(0).total_memory / (1024 ** 3)
        except Exception:
            return "cuda"
        if (
            "jina-embeddings-v3" in self.config.embedding_model_name.lower()
            and total_gb < self.config.embedding_auto_cpu_max_gpu_memory_gb
        ):
            LOGGER.warning(
                "GPU VRAM %.2f GiB is below the %.2f GiB safety threshold for %s. Using CPU embeddings.",
                total_gb,
                self.config.embedding_auto_cpu_max_gpu_memory_gb,
                self.config.embedding_model_name,
            )
            return "cpu"
        return "cuda"

    @staticmethod
    def _cleanup_jina_transformers_cache() -> None:
        cache_root = Path(os.environ.get("HF_HOME", str(Path.home() / ".cache" / "huggingface")))
        modules_dir = cache_root / "modules" / "transformers_modules" / "jinaai"
        if modules_dir.exists():
            shutil.rmtree(modules_dir, ignore_errors=True)

    @staticmethod
    def _is_cuda_oom(error: BaseException) -> bool:
        message = str(error).lower()
        return "cuda" in message and "out of memory" in message

    @staticmethod
    def _clear_embedding_memory() -> None:
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()

    def _load_sentence_transformer(self, model_name: str) -> SentenceTransformer:
        kwargs = {"trust_remote_code": True}
        if self._force_cpu_for_embeddings:
            kwargs["device"] = "cpu"
        model = SentenceTransformer(model_name, **kwargs)
        if hasattr(model, "max_seq_length"):
            model.max_seq_length = self.config.embedding_max_seq_length
        return model

    @property
    def model(self) -> SentenceTransformer:
        if self._model is None:
            try:
                self._model = self._load_sentence_transformer(self.config.embedding_model_name)
            except ModuleNotFoundError as error:
                error_text = str(error)
                if "transformers_modules.jinaai" not in error_text:
                    raise

                LOGGER.warning(
                    "Jina remote module import failed (%s). Clearing module cache and retrying once.",
                    error,
                )
                self._cleanup_jina_transformers_cache()
                try:
                    self._model = self._load_sentence_transformer(self.config.embedding_model_name)
                except ModuleNotFoundError as retry_error:
                    LOGGER.warning(
                        "Jina remote module retry failed (%s). Falling back to %s.",
                        retry_error,
                        self.config.embedding_fallback_model_name,
                    )
                    fallback_kwargs: dict[str, str] = {}
                    if self._force_cpu_for_embeddings:
                        fallback_kwargs["device"] = "cpu"
                    self._model = SentenceTransformer(self.config.embedding_fallback_model_name, **fallback_kwargs)
                    if hasattr(self._model, "max_seq_length"):
                        self._model.max_seq_length = self.config.embedding_max_seq_length
        return self._model

    def encode_passages(self, texts: Sequence[str]) -> np.ndarray:
        return self._encode(list(texts), self.config.embedding_passage_task)

    def encode_queries(self, texts: Sequence[str]) -> np.ndarray:
        normalized_queries = [
            self.config.embedding_query_prefix + self.normalizer.normalize_for_embedding(text)
            for text in texts
        ]
        return self._encode(normalized_queries, self.config.embedding_query_task)

    def _encode(self, texts: list[str], task: str) -> np.ndarray:
        if not texts:
            return np.empty((0, self.config.embedding_dimension), dtype=np.float32)

        model = self.model
        if hasattr(model, "max_seq_length"):
            model.max_seq_length = self.config.embedding_max_seq_length

        active_batch_size = max(
            self.config.embedding_min_batch_size,
            min(self.config.embedding_batch_size, len(texts)),
        )
        use_cpu = self._force_cpu_for_embeddings or not torch.cuda.is_available()

        while True:
            kwargs = {
                "batch_size": active_batch_size,
                "show_progress_bar": True,
                "convert_to_numpy": True,
                "normalize_embeddings": True,
            }
            if use_cpu:
                model.to("cpu")
                kwargs["device"] = "cpu"

            try:
                try:
                    vectors = model.encode(
                        texts,
                        task=task,
                        truncate_dim=self.config.embedding_dimension,
                        **kwargs,
                    )
                except TypeError:
                    vectors = model.encode(texts, **kwargs)
                break
            except (RuntimeError, torch.OutOfMemoryError) as error:
                if use_cpu or not self._is_cuda_oom(error):
                    raise

                self._clear_embedding_memory()
                if active_batch_size > self.config.embedding_min_batch_size:
                    active_batch_size = max(self.config.embedding_min_batch_size, active_batch_size // 2)
                    LOGGER.warning(
                        "CUDA OOM while embedding %s texts. Retrying with batch_size=%s and max_seq_length=%s.",
                        len(texts),
                        active_batch_size,
                        self.config.embedding_max_seq_length,
                    )
                    continue

                LOGGER.warning(
                    "CUDA OOM persisted at batch_size=%s. Falling back to CPU embeddings for the rest of the build.",
                    active_batch_size,
                )
                self._force_cpu_for_embeddings = True
                use_cpu = True
                self._clear_embedding_memory()

        vectors = np.asarray(vectors, dtype=np.float32)
        if vectors.ndim != 2:
            raise ValueError(f"Embedding output must be 2D, got shape={vectors.shape}")
        if vectors.shape[0] != len(texts):
            raise ValueError(f"Embedding row count mismatch: expected {len(texts)}, got {vectors.shape[0]}")
        if vectors.shape[1] != self.config.embedding_dimension:
            raise ValueError(
                f"Embedding dimension mismatch: expected {self.config.embedding_dimension}, got {vectors.shape[1]}"
            )
        return vectors


class ChromaIndexer:
    """Stream embeddings into Chroma in bounded-memory batches."""

    def __init__(self, config: PipelineConfig):
        self.config = config

    def build(self, chunks: Sequence[DocumentChunk], encoder: EmbeddingModel) -> chromadb.Collection:
        if self.config.reset_chroma_directory and self.config.chroma_dir.exists():
            try:
                shutil.rmtree(self.config.chroma_dir)
            except PermissionError as error:
                fallback_name = f"{self.config.chroma_directory_name}_{int(time.time())}"
                LOGGER.warning(
                    "Could not remove locked Chroma directory '%s' (%s). Using fallback directory '%s'.",
                    self.config.chroma_dir,
                    error,
                    fallback_name,
                )
                self.config.chroma_directory_name = fallback_name
        self.config.chroma_dir.mkdir(parents=True, exist_ok=True)

        client = chromadb.PersistentClient(path=str(self.config.chroma_dir))
        existing = {item.name if hasattr(item, "name") else str(item) for item in client.list_collections()}
        if self.config.chroma_collection_name in existing:
            try:
                client.delete_collection(self.config.chroma_collection_name)
            except Exception as error:
                LOGGER.warning(
                    "Could not delete existing collection '%s' (continuing with upsert): %s",
                    self.config.chroma_collection_name,
                    error,
                )

        collection = client.get_or_create_collection(
            self.config.chroma_collection_name,
            metadata={"hnsw:space": "cosine"},
        )

        for batch in batched(chunks, self.config.chroma_upsert_batch_size):
            embeddings = encoder.encode_passages([chunk.text_for_embedding for chunk in batch])
            collection.upsert(
                ids=[chunk.chunk_id for chunk in batch],
                documents=[chunk.text for chunk in batch],
                metadatas=[{"chunk_id": chunk.chunk_id, **chunk.metadata.to_chroma()} for chunk in batch],
                embeddings=embeddings.tolist(),
            )
        return collection


class HybridRetriever:
    """Accuracy-oriented hybrid retriever with exact ayah-ref fallback."""

    def __init__(self, collection: chromadb.Collection, encoder: EmbeddingModel, normalizer: ArabicTextNormalizer):
        self.collection = collection
        self.encoder = encoder
        self.normalizer = normalizer
        self.rows = self.collection.get(include=["documents", "metadatas"])
        self.search_rows = [
            {
                "idx": idx,
                "chunk_id": chunk_id,
                "text": document,
                "meta": metadata,
                "ref": metadata["ayah_ref"],
            }
            for idx, (chunk_id, document, metadata) in enumerate(
                zip(self.rows["ids"], self.rows["documents"], self.rows["metadatas"])
            )
        ]

    @staticmethod
    def _build_where(
        *,
        language: str | None = "ar",
        surah_number: int | None = None,
        juz: int | None = None,
        content_type: str | None = None,
        edition_identifier: str | None = None,
        ayah_ref: str | None = None,
    ) -> dict[str, Any] | None:
        conditions: list[dict[str, Any]] = []
        if language:
            conditions.append({"language": {"$eq": language}})
        if surah_number is not None:
            conditions.append({"surah_number": {"$eq": surah_number}})
        if juz is not None:
            conditions.append({"juz": {"$eq": juz}})
        if content_type:
            conditions.append({"content_type": {"$eq": content_type}})
        if edition_identifier:
            conditions.append({"edition_identifier": {"$eq": edition_identifier}})
        if ayah_ref:
            conditions.append({"ayah_ref": {"$eq": ayah_ref}})
        if not conditions:
            return None
        if len(conditions) == 1:
            return conditions[0]
        return {"$and": conditions}

    @staticmethod
    def _row_matches(
        row: dict[str, Any],
        *,
        language: str | None = "ar",
        surah_number: int | None = None,
        juz: int | None = None,
        content_type: str | None = None,
        edition_identifier: str | None = None,
    ) -> bool:
        meta = row["meta"]
        if language and meta.get("language") != language:
            return False
        if surah_number is not None and meta.get("surah_number") != surah_number:
            return False
        if juz is not None and meta.get("juz") != juz:
            return False
        if content_type and meta.get("content_type") != content_type:
            return False
        if edition_identifier and meta.get("edition_identifier") != edition_identifier:
            return False
        return True

    def exact_ayah_lookup(
        self,
        query: str,
        *,
        edition_identifier: str = "quran-uthmani",
    ) -> list[dict[str, Any]]:
        ayah_ref = self.normalizer.normalize_ayah_ref(query)
        if not ayah_ref:
            return []
        rows = self.collection.get(
            where=self._build_where(
                ayah_ref=ayah_ref,
                content_type="quran_ayah",
                edition_identifier=edition_identifier,
            ),
            include=["documents", "metadatas"],
        )
        output: list[dict[str, Any]] = []
        for chunk_id, text, meta in zip(rows.get("ids", []), rows.get("documents", []), rows.get("metadatas", [])):
            output.append(
                {
                    "chunk_id": chunk_id,
                    "text": text,
                    "meta": meta,
                    "ref": meta["ayah_ref"],
                    "score": 1.0,
                    "method": "exact_ref",
                }
            )
        return output

    def exact_text_lookup(
        self,
        query: str,
        *,
        language: str | None = "ar",
        content_type: str | None = None,
        edition_identifier: str | None = None,
        top_k: int = 5,
    ) -> list[dict[str, Any]]:
        normalized_query = self.normalizer.normalize_for_bm25_query(query)
        if not normalized_query:
            return []
        query_tokens = self.normalizer.tokenize_query_for_bm25(query)
        allow_phrase_match = len(query_tokens) >= 3 and len(normalized_query) >= 12
        hits: list[dict[str, Any]] = []
        for row in self.search_rows:
            if not self._row_matches(
                row,
                language=language,
                content_type=content_type,
                edition_identifier=edition_identifier,
            ):
                continue
            normalized_text = row["meta"].get("bm25_normalized_text", "")
            is_exact = normalized_query == normalized_text
            is_phrase = allow_phrase_match and f" {normalized_query} " in f" {normalized_text} "
            if is_exact or is_phrase:
                hits.append(
                    {
                        "chunk_id": row["chunk_id"],
                        "text": row["text"],
                        "meta": row["meta"],
                        "ref": row["ref"],
                        "score": 1.0 if is_exact else 0.96,
                        "method": "exact_text" if is_exact else "phrase_text",
                    }
                )
                if len(hits) >= top_k:
                    break
        return hits

    def dense_search(
        self,
        query: str,
        *,
        top_k: int = 10,
        language: str | None = "ar",
        surah_number: int | None = None,
        juz: int | None = None,
        content_type: str | None = None,
        edition_identifier: str | None = None,
    ) -> list[dict[str, Any]]:
        query_vector = self.encoder.encode_queries([query])[0]
        response = self.collection.query(
            query_embeddings=[query_vector.tolist()],
            n_results=top_k,
            where=self._build_where(
                language=language,
                surah_number=surah_number,
                juz=juz,
                content_type=content_type,
                edition_identifier=edition_identifier,
            ),
            include=["documents", "metadatas", "distances"],
        )

        results: list[dict[str, Any]] = []
        for chunk_id, document, metadata, distance in zip(
            response.get("ids", [[]])[0],
            response.get("documents", [[]])[0],
            response.get("metadatas", [[]])[0],
            response.get("distances", [[]])[0],
        ):
            metadata = metadata or {}
            results.append(
                {
                    "chunk_id": chunk_id,
                    "text": document,
                    "meta": metadata,
                    "ref": metadata["ayah_ref"],
                    "score": 1.0 - float(distance),
                    "method": "dense",
                }
            )
        return results

    def bm25_search(
        self,
        query: str,
        *,
        top_k: int = 10,
        language: str | None = "ar",
        surah_number: int | None = None,
        juz: int | None = None,
        content_type: str | None = None,
        edition_identifier: str | None = None,
    ) -> list[dict[str, Any]]:
        # Safety choice: lexical scoring is scoped to one content type to avoid
        # cross-contamination between canonical Quran ayahs and tafsir passages.
        effective_content_type = content_type or "quran_ayah"
        filtered_rows = [
            row
            for row in self.search_rows
            if self._row_matches(
                row,
                language=language,
                surah_number=surah_number,
                juz=juz,
                content_type=effective_content_type,
                edition_identifier=edition_identifier,
            )
        ]
        if not filtered_rows:
            return []

        tokenized_corpus = [
            self.normalizer.tokenize_document_for_bm25(row["meta"]["bm25_normalized_text"])
            for row in filtered_rows
        ]
        bm25 = BM25Okapi(tokenized_corpus)
        query_tokens = self.normalizer.tokenize_query_for_bm25(query)
        scores = bm25.get_scores(query_tokens)
        normalized_query = self.normalizer.normalize_for_bm25_query(query)

        results: list[dict[str, Any]] = []
        for row, score in zip(filtered_rows, scores):
            if score <= 0:
                continue
            boosted_score = float(score)
            normalized_text = row["meta"].get("bm25_normalized_text", "")
            if normalized_query and normalized_query in normalized_text:
                boosted_score += 1.5
            results.append(
                {
                    "chunk_id": row["chunk_id"],
                    "text": row["text"],
                    "meta": row["meta"],
                    "ref": row["ref"],
                    "score": boosted_score,
                    "method": "bm25",
                }
            )

        results.sort(key=lambda item: item["score"], reverse=True)
        return results[:top_k]

    def hybrid_search(
        self,
        query: str,
        *,
        top_k: int = 5,
        language: str | None = "ar",
        surah_number: int | None = None,
        juz: int | None = None,
        content_type: str | None = None,
        edition_identifier: str | None = None,
    ) -> list[dict[str, Any]]:
        exact_ref = self.exact_ayah_lookup(query, edition_identifier=edition_identifier or "quran-uthmani")
        if exact_ref and (content_type in (None, "quran_ayah")):
            return exact_ref[:top_k]

        exact_text = self.exact_text_lookup(
            query,
            language=language,
            content_type=content_type,
            edition_identifier=edition_identifier,
            top_k=top_k,
        )
        if exact_text:
            return exact_text[:top_k]

        dense_results = self.dense_search(
            query,
            top_k=20,
            language=language,
            surah_number=surah_number,
            juz=juz,
            content_type=content_type,
            edition_identifier=edition_identifier,
        )
        bm25_results = self.bm25_search(
            query,
            top_k=20,
            language=language,
            surah_number=surah_number,
            juz=juz,
            content_type=content_type,
            edition_identifier=edition_identifier,
        )

        fused: dict[str, dict[str, Any]] = {}
        for method_name, result_set in [("dense", dense_results), ("bm25", bm25_results)]:
            for rank, item in enumerate(result_set, start=1):
                entry = fused.setdefault(
                    item["chunk_id"],
                    {
                        "chunk_id": item["chunk_id"],
                        "text": item["text"],
                        "meta": item["meta"],
                        "ref": item["ref"],
                        "score": 0.0,
                        "methods": [],
                    },
                )
                entry["score"] += 1.0 / (60 + rank)
                entry["methods"].append(method_name)

        return sorted(fused.values(), key=lambda item: item["score"], reverse=True)[:top_k]


class CorpusValidator:
    """Integrity checks for payloads, chunks, Chroma, and retrieval behavior."""

    def __init__(self, config: PipelineConfig, normalizer: ArabicTextNormalizer):
        self.config = config
        self.normalizer = normalizer

    def validate_chunks(self, chunks: Sequence[DocumentChunk]) -> dict[str, Any]:
        expected = Counter((spec.content_type, spec.language, spec.identifier) for spec in self.config.edition_specs)
        expected = Counter({key: value * EXPECTED_QURAN_AYAH_TOTAL for key, value in expected.items()})

        actual = Counter((chunk.metadata.content_type, chunk.metadata.language, chunk.metadata.edition_identifier) for chunk in chunks)
        if actual != expected:
            raise ValueError(f"Chunk composition mismatch: {actual} != {expected}")

        chunk_ids = [chunk.chunk_id for chunk in chunks]
        if len(set(chunk_ids)) != len(chunk_ids):
            raise ValueError("Duplicate chunk IDs detected")

        for chunk in chunks:
            expected_hash = hashlib.sha256(chunk.text.encode("utf-8")).hexdigest()
            if chunk.metadata.canonical_text_sha256 != expected_hash:
                raise ValueError(f"Canonical hash mismatch for {chunk.chunk_id}")

            if chunk.metadata.raw_source_text != chunk.text:
                raise ValueError(f"Raw source text mismatch for {chunk.chunk_id}")
            if chunk.metadata.canonical_text != chunk.text:
                raise ValueError(f"Canonical text mismatch for {chunk.chunk_id}")

            expected_id = self._expected_chunk_id(
                chunk.metadata.content_type,
                chunk.metadata.edition_identifier,
                chunk.metadata.surah_number,
                chunk.metadata.ayah_number_in_surah,
            )
            if chunk.chunk_id != expected_id:
                raise ValueError(
                    f"Non-deterministic chunk ID for {chunk.metadata.ayah_ref}/{chunk.metadata.edition_identifier}: "
                    f"expected {expected_id}, got {chunk.chunk_id}"
                )

            expected_fingerprint = QuranChunkBuilder._build_metadata_fingerprint(
                chunk.metadata.edition_identifier,
                chunk.metadata.ayah_ref,
                chunk.metadata.canonical_text_sha256,
                chunk.metadata.source_url,
            )
            if chunk.metadata.metadata_fingerprint != expected_fingerprint:
                raise ValueError(f"Metadata fingerprint mismatch for {chunk.chunk_id}")

        refs = {(chunk.metadata.edition_identifier, chunk.metadata.ayah_ref) for chunk in chunks}
        if len(refs) != len(chunks):
            raise ValueError("Duplicate edition+ayah references detected")

        canonical_refs = {
            chunk.metadata.ayah_ref
            for chunk in chunks
            if chunk.metadata.edition_identifier == "quran-uthmani"
        }
        if len(canonical_refs) != EXPECTED_QURAN_AYAH_TOTAL:
            raise ValueError("Canonical Quran reference set is incomplete")

        for spec in self.config.edition_specs:
            edition_refs = {
                chunk.metadata.ayah_ref
                for chunk in chunks
                if chunk.metadata.edition_identifier == spec.identifier
            }
            if edition_refs != canonical_refs:
                missing = sorted(canonical_refs - edition_refs)[:5]
                extras = sorted(edition_refs - canonical_refs)[:5]
                raise ValueError(
                    f"Edition {spec.identifier} ayah references do not match canonical Quran set. "
                    f"Missing sample={missing}, extra sample={extras}"
                )

        composition_str_keys = {
            f"{ct}_{lang}_{id}": count
            for (ct, lang, id), count in actual.items()
        }

        return {
            "chunk_count": len(chunks),
            "composition": composition_str_keys,
            "canonical_ref_count": len(canonical_refs),
            "unique_chunk_ids": len(set(chunk_ids)),
        }

    def validate_collection(self, collection: chromadb.Collection, encoder: EmbeddingModel) -> dict[str, Any]:
        expected_count = len(self.config.edition_specs) * EXPECTED_QURAN_AYAH_TOTAL
        if collection.count() != expected_count:
            raise ValueError(f"Unexpected collection size: expected {expected_count}, got {collection.count()}")

        rows = collection.get(include=["documents", "metadatas"])
        ids = rows.get("ids", [])
        metadatas = rows.get("metadatas", [])
        documents = rows.get("documents", [])
        if len(ids) != expected_count or len(set(ids)) != expected_count:
            raise ValueError("Collection IDs are missing or duplicated")

        required_metadata = {
            "chunk_id",
            "content_type",
            "source",
            "source_family",
            "edition_identifier",
            "edition_name",
            "language",
            "surah_number",
            "ayah_number_in_surah",
            "ayah_number_global",
            "ayah_ref",
            "surah_name_arabic",
            "juz",
            "page",
            "ruku",
            "manzil",
            "hizb_quarter",
            "sajda",
            "raw_source_text",
            "canonical_text",
            "bm25_normalized_text",
            "embedding_normalized_text",
            "canonical_text_sha256",
            "normalization_profile",
            "chunk_schema_version",
            "pipeline_version",
            "ingested_at_utc",
            "metadata_fingerprint",
            "source_url",
        }

        for idx, (meta, document) in enumerate(zip(metadatas, documents)):
            missing = required_metadata - set(meta)
            if missing:
                raise ValueError(f"Metadata row {idx} missing fields: {sorted(missing)}")
            if meta["source"] not in {"quran", "tafsir"}:
                raise ValueError(f"Invalid machine-stable source field in row {idx}: {meta['source']}")
            if meta["raw_source_text"] != document or meta["canonical_text"] != document:
                raise ValueError(f"Canonical/raw mismatch in row {idx}")

        canonical_quran_refs = {
            meta["ayah_ref"]
            for meta in metadatas
            if meta["content_type"] == "quran_ayah" and meta["edition_identifier"] == "quran-uthmani"
        }

        for spec in self.config.edition_specs:
            edition_refs = {
                meta["ayah_ref"]
                for meta in metadatas
                if meta["edition_identifier"] == spec.identifier
            }
            if edition_refs != canonical_quran_refs:
                raise ValueError(f"Collection edition alignment failed for {spec.identifier}")

        retriever = HybridRetriever(collection, encoder, self.normalizer)
        probe_results = retriever.hybrid_search("قل هو الله احد", top_k=5, content_type="quran_ayah")
        refs = [item["ref"] for item in probe_results]
        if "112:1" not in refs:
            raise ValueError(f"Hybrid probe failed for Surah Al-Ikhlas: {refs}")

        exact = retriever.exact_ayah_lookup("2:255")
        if not exact or exact[0]["ref"] != "2:255":
            raise ValueError("Exact ayah-ref lookup failed for 2:255")

        exact_ar_digits = retriever.exact_ayah_lookup("٢:٢٥٥")
        if not exact_ar_digits or exact_ar_digits[0]["ref"] != "2:255":
            raise ValueError("Exact ayah-ref lookup failed for Arabic-Indic digits")

        return {
            "stored_count": collection.count(),
            "probe_refs": refs,
            "exact_lookup_ok": True,
        }

    @staticmethod
    def _expected_chunk_id(
        content_type: str,
        edition_identifier: str,
        surah_number: int,
        ayah_number_in_surah: int,
    ) -> str:
        if content_type == "quran_ayah" and edition_identifier == "quran-uthmani":
            return f"quran_ar_{surah_number}_{ayah_number_in_surah}"
        safe_edition = edition_identifier.replace(".", "_").replace("-", "_")
        return f"{content_type}_{safe_edition}_ar_{surah_number}_{ayah_number_in_surah}"


class QuranRAGPipeline:
    """Orchestrate the full ingestion pipeline."""

    def __init__(self, config: PipelineConfig | None = None):
        self.config = config or PipelineConfig()
        self.normalizer = ArabicTextNormalizer()
        self.client = QuranHubClient(self.config)
        self.builder = QuranChunkBuilder(self.config, self.normalizer)
        self.encoder = EmbeddingModel(self.config, self.normalizer)
        self.indexer = ChromaIndexer(self.config)
        self.validator = CorpusValidator(self.config, self.normalizer)

    def run(self) -> dict[str, Any]:
        payloads = self.client.fetch_all()
        chunks = self.builder.build(payloads)
        chunk_report = self.validator.validate_chunks(chunks)
        collection = self.indexer.build(chunks, self.encoder)
        validation_report = self.validator.validate_collection(collection, self.encoder)

        corpus_fingerprint = hashlib.sha256(
            "\n".join(
                sorted(
                    f"{chunk.chunk_id}|{chunk.metadata.edition_identifier}|{chunk.metadata.ayah_ref}|{chunk.metadata.canonical_text_sha256}"
                    for chunk in chunks
                )
            ).encode("utf-8")
        ).hexdigest()

        manifest = {
            "pipeline_version": PIPELINE_VERSION,
            "chunk_schema_version": CHUNK_SCHEMA_VERSION,
            "build_timestamp_utc": datetime.now(timezone.utc).isoformat(),
            "api_base_url": self.config.api_base_url,
            "editions": [spec.identifier for spec in self.config.edition_specs],
            "embedding_model_name": self.config.embedding_model_name,
            "embedding_dimension": self.config.embedding_dimension,
            "collection_name": self.config.chroma_collection_name,
            "corpus_fingerprint": corpus_fingerprint,
            "normalization_policy": {
                "canonical_text": "stored verbatim",
                "bm25_document": "conservative; no risky orthographic folding",
                "bm25_query": "cautious alef-family folding",
                "dense": "light normalization with alef-family fold",
                "risky_rules_disabled": ["teh_marbuta_to_heh", "alef_maqsura_to_ya", "aggressive_uthmani_rewrites"],
            },
            "chunk_report": chunk_report,
            "validation_report": validation_report,
        }

        self.config.manifest_path.write_text(
            json.dumps(manifest, ensure_ascii=False, indent=2),
            encoding="utf-8",
        )

        if self.config.export_zip_path.exists():
            self.config.export_zip_path.unlink()
        archive_path = shutil.make_archive(
            str(self.config.export_zip_path.with_suffix("")),
            "zip",
            root_dir=str(self.config.chroma_dir.parent),
            base_dir=self.config.chroma_dir.name,
        )
        manifest["archive_path"] = archive_path


2026-03-14 14:15:07.143168: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1773497707.377994     172 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1773497707.442212     172 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1773497707.969325     172 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1773497707.969376     172 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1773497707.969379     172 computation_placer.cc:177] computation placer alr

In [2]:
logging.basicConfig(level=logging.INFO, format="%(levelname)s %(name)s: %(message)s")

config = PipelineConfig(
    workspace_dir=Path("/kaggle/working/quran_rag_build"),
    refresh_api_cache=False,
    reset_chroma_directory=True,
    embedding_batch_size=2,
    embedding_max_seq_length=300,
    embedding_device="cuda",
    embedding_auto_cpu_max_gpu_memory_gb=20.0,
    chroma_upsert_batch_size=128,
)

print("Workspace:", config.workspace_dir)
print("Selected editions:", [spec.identifier for spec in config.edition_specs])
print("Embedding model:", config.embedding_model_name)
print("Embedding batch size:", config.embedding_batch_size)
print("Embedding max sequence length:", config.embedding_max_seq_length)
print("Embedding device policy:", config.embedding_device)
print("Auto CPU threshold (GiB):", config.embedding_auto_cpu_max_gpu_memory_gb)


Workspace: /kaggle/working/quran_rag_build
Selected editions: ['quran-uthmani', 'ar.tabari', 'ar.muyassar', 'ar.mukhtasar']
Embedding model: jinaai/jina-embeddings-v3
Embedding batch size: 2
Embedding max sequence length: 300
Embedding device policy: cuda
Auto CPU threshold (GiB): 20.0


In [3]:
import torch

gpu_vram_gb = None
if torch.cuda.is_available():
    gpu_vram_gb = round(torch.cuda.get_device_properties(0).total_memory / (1024 ** 3), 2)

print(
    "Embedding runtime safeguards:",
    {
        "batch_size": config.embedding_batch_size,
        "min_batch_size": config.embedding_min_batch_size,
        "max_seq_length": config.embedding_max_seq_length,
        "device_policy": config.embedding_device,
        "auto_cpu_threshold_gb": config.embedding_auto_cpu_max_gpu_memory_gb,
        "cuda_available": torch.cuda.is_available(),
        "gpu_vram_gb": gpu_vram_gb,
        "allocator": os.environ.get("PYTORCH_CUDA_ALLOC_CONF"),
    },
)


Embedding runtime safeguards: {'batch_size': 2, 'min_batch_size': 1, 'max_seq_length': 300, 'device_policy': 'cuda', 'auto_cpu_threshold_gb': 20.0, 'cuda_available': True, 'gpu_vram_gb': 14.56, 'allocator': 'expandable_segments:True'}


## Build The Chroma Artifact

This cell runs the full pipeline end-to-end and creates:
- the persisted Chroma directory `quran_full`
- the zip artifact `quran_full.zip`
- a JSON build manifest


In [4]:
import tempfile
from pathlib import Path
from datetime import datetime


def _is_writable_directory(path: Path) -> bool:
    try:
        path.mkdir(parents=True, exist_ok=True)
        probe = path / ".write_probe"
        probe.write_text("ok", encoding="utf-8")
        probe.unlink(missing_ok=True)
        return True
    except Exception:
        return False


def _make_recovery_config(base: PipelineConfig, workspace_dir: Path) -> PipelineConfig:
    return PipelineConfig(
        workspace_dir=workspace_dir,
        api_base_url=base.api_base_url,
        embedding_model_name=base.embedding_model_name,
        embedding_fallback_model_name=base.embedding_fallback_model_name,
        embedding_dimension=base.embedding_dimension,
        embedding_batch_size=base.embedding_batch_size,
        embedding_min_batch_size=base.embedding_min_batch_size,
        embedding_max_seq_length=base.embedding_max_seq_length,
        embedding_device=base.embedding_device,
        embedding_auto_cpu_max_gpu_memory_gb=base.embedding_auto_cpu_max_gpu_memory_gb,
        embedding_query_task=base.embedding_query_task,
        embedding_passage_task=base.embedding_passage_task,
        embedding_query_prefix=base.embedding_query_prefix,
        embedding_passage_prefix=base.embedding_passage_prefix,
        chroma_collection_name=base.chroma_collection_name,
        chroma_directory_name=base.chroma_directory_name,
        export_zip_name=base.export_zip_name,
        request_timeout_seconds=base.request_timeout_seconds,
        request_max_retries=base.request_max_retries,
        request_backoff_seconds=base.request_backoff_seconds,
        refresh_api_cache=base.refresh_api_cache,
        reset_chroma_directory=base.reset_chroma_directory,
        chroma_upsert_batch_size=base.chroma_upsert_batch_size,
        edition_specs=base.edition_specs,
    )


def _is_storage_write_error(error: Exception) -> bool:
    message = str(error).lower()
    tokens = (
        "readonly",
        "read-only",
        "permission denied",
        "attempt to write a readonly database",
        "database is locked",
        "winerror 32",
        "used by another process",
        "cannot access the file because it is being used by another process",
    )
    return any(token in message for token in tokens)


candidates = [
    config.workspace_dir,
    Path("/kaggle/working/quran_rag_build"),
    Path("/tmp/quran_rag_kaggle_build"),
]

unique_candidates: list[Path] = []
seen = set()
for candidate in candidates:
    key = str(candidate)
    if key in seen:
        continue
    seen.add(key)
    unique_candidates.append(candidate)

if not any(_is_writable_directory(candidate) for candidate in unique_candidates):
    fallback = Path(tempfile.mkdtemp(prefix="quran_rag_build_"))
    unique_candidates.append(fallback)

last_error: Exception | None = None
active_config = config

for candidate in unique_candidates:
    if not _is_writable_directory(candidate):
        continue

    active_config = _make_recovery_config(config, candidate)
    print(f"Trying build workspace: {active_config.workspace_dir}")
    try:
        pipeline = QuranRAGPipeline(active_config)
        build_report = pipeline.run()
        break
    except Exception as error:
        last_error = error
        if _is_storage_write_error(error):
            print(f"Workspace failed due to storage write issue: {error}")
            continue
        raise
else:
    raise RuntimeError(f"Failed to build Chroma artifact in all candidate workspaces. Last error: {last_error}")

config = active_config
collection = chromadb.PersistentClient(path=str(config.chroma_dir)).get_collection(config.chroma_collection_name)
normalizer = pipeline.normalizer
embedding_model = pipeline.encoder
SELECTED_EDITIONS = [spec.identifier for spec in config.edition_specs]
EXPECTED_COUNTS = {
    (spec.content_type, spec.language, spec.identifier): spec.expected_ayah_count
    for spec in config.edition_specs
}
EXPECTED_TOTAL_CHUNKS = sum(EXPECTED_COUNTS.values())
EXPORT_ZIP_PATH = config.export_zip_path

print(json.dumps(build_report, ensure_ascii=False, indent=2))
print(f"Build completed at: {datetime.now().isoformat()}")


Trying build workspace: /kaggle/working/quran_rag_build


modules.json:   0%|          | 0.00/378 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/464 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

custom_st.py: 0.00B [00:00, ?B/s]

...

Batches:   0%|          | 0/64 [00:00<?, ?it/s]

Batches:   0%|          | 0/64 [00:00<?, ?it/s]

Batches:   0%|          | 0/56 [00:00<?, ?it/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

null
Build completed at: 2026-03-14T14:50:54.965298


## Large Validation Cell

This validation section is designed for high-stakes Qur'anic text integrity and retrieval precision.

It validates:
- exact corpus size and edition composition
- duplicate detection and canonical ayah coverage
- deterministic chunk IDs and canonical text SHA-256 hashes
- metadata completeness and schema expectations
- exact ayah-reference lookup (`2:255`, `٢:٢٥٥`)
- near-exact lookup without tashkeel
- hybrid retrieval behavior (dense + BM25 + RRF)
- metadata filtering for Quran vs tafsir and edition-specific search
- conservative normalization constraints (no `ة↔ه`, no `ى↔ي` collapsing)

## Qur'anic Normalization Audit Matrix

This matrix documents what is allowed in this notebook and where it is applied.

| Rule | Classification | Canonical Storage | BM25 Document | BM25 Query | Dense Query/Passage |
|---|---|---|---|---|---|
| Unicode canonicalization + zero-width cleanup | Context-dependent | ❌ | ✅ | ✅ | ✅ |
| Diacritic removal | Context-dependent | ❌ | ✅ | ✅ | ✅ |
| Tatweel removal | Safe | ❌ | ✅ | ✅ | ✅ |
| Alef-family fold (`أ/إ/آ/ٱ → ا`) | Context-dependent | ❌ | ⚠️ limited via token aliasing | ✅ | ✅ |
| `ة → ه` | Risky | ❌ | ❌ | ❌ | ❌ |
| `ى → ي` | Risky | ❌ | ❌ | ❌ | ❌ |
| Aggressive Uthmani rewriting | Risky | ❌ | ❌ | ❌ | ❌ |

Policy summary:
- Canonical Qur'an text remains untouched in `documents`.
- Retrieval-specific forms are stored only in derived metadata fields.
- High-risk folds are intentionally disabled to protect citation precision.

In [7]:
import hashlib
import numpy as np
from collections import Counter, defaultdict
from typing import Any

# ── 0. Init guard ─────────────────────────────────────────────────────────────
if "collection" not in globals():
    normalizer      = ArabicTextNormalizer()
    embedding_model = EmbeddingModel(config, normalizer)
    collection      = chromadb.PersistentClient(
        path=str(config.chroma_dir)
    ).get_collection(config.chroma_collection_name)
    SELECTED_EDITIONS = [spec.identifier for spec in config.edition_specs]
    EXPECTED_COUNTS   = {
        (spec.content_type, spec.language, spec.identifier): spec.expected_ayah_count
        for spec in config.edition_specs
    }
    EXPECTED_TOTAL_CHUNKS = sum(EXPECTED_COUNTS.values())
    EXPORT_ZIP_PATH       = config.export_zip_path

if "retriever" not in globals():
    retriever = HybridRetriever(collection, embedding_model, normalizer)

PASS = "✓ PASS"
FAIL = "✗ FAIL"
_failures: list[str] = []

def check(label: str, condition: bool, detail: str = "") -> None:
    if condition:
        print(f"  {PASS}  {label}")
    else:
        msg = f"{label}" + (f" — {detail}" if detail else "")
        print(f"  {FAIL}  {msg}")
        _failures.append(msg)

def section(title: str) -> None:
    print(f"\n{'─'*60}")
    print(f"  {title}")
    print(f"{'─'*60}")

# ═══════════════════════════════════════════════════════════════
# 1. COLLECTION INTEGRITY
# ═══════════════════════════════════════════════════════════════
section("1 · Collection integrity")

stored_count = collection.count()
check("Total chunk count matches expected",
      stored_count == EXPECTED_TOTAL_CHUNKS,
      f"got {stored_count}, expected {EXPECTED_TOTAL_CHUNKS}")

all_records   = collection.get(
    limit=EXPECTED_TOTAL_CHUNKS,
    include=["documents", "metadatas", "embeddings"],
)
all_ids       = all_records["ids"]
all_documents = all_records["documents"]
all_metadatas = all_records["metadatas"]
all_embeddings = all_records["embeddings"]   # list of lists

check("IDs count",       len(all_ids)        == EXPECTED_TOTAL_CHUNKS)
check("Docs count",      len(all_documents)  == EXPECTED_TOTAL_CHUNKS)
check("Metadata count",  len(all_metadatas)  == EXPECTED_TOTAL_CHUNKS)
check("Embeddings count",len(all_embeddings) == EXPECTED_TOTAL_CHUNKS)
check("No duplicate IDs",
      len(set(all_ids)) == EXPECTED_TOTAL_CHUNKS,
      f"{EXPECTED_TOTAL_CHUNKS - len(set(all_ids))} duplicates found")
check("No empty documents",
      all(text is not None and str(text).strip() != "" for text in all_documents))
check("All language=ar",
      all(m["language"] == "ar" for m in all_metadatas))
check("Edition set matches config",
      set(m["edition_identifier"] for m in all_metadatas) == set(SELECTED_EDITIONS))
check("Source field only quran/tafsir",
      set(m["source"] for m in all_metadatas) <= {"quran", "tafsir"})

# ═══════════════════════════════════════════════════════════════
# 2. EMBEDDING VECTOR SANITY
# ═══════════════════════════════════════════════════════════════
section("2 · Embedding vector sanity")

emb_matrix = np.array(all_embeddings, dtype=np.float32)

check("Embedding matrix is 2-D",
      emb_matrix.ndim == 2,
      f"got shape {emb_matrix.shape}")
check(f"Embedding dimension = {config.embedding_dimension}",
      emb_matrix.shape[1] == config.embedding_dimension,
      f"got {emb_matrix.shape[1]}")
check("No NaN values in any embedding",
      not np.isnan(emb_matrix).any())
check("No all-zero embeddings",
      not np.any(np.all(emb_matrix == 0, axis=1)))

norms = np.linalg.norm(emb_matrix, axis=1)
check("All embeddings are unit-normalised (norm ∈ [0.99, 1.01])",
      bool(np.all((norms >= 0.99) & (norms <= 1.01))),
      f"min={norms.min():.4f}  max={norms.max():.4f}")

# Intra-corpus cosine stats (random 500-sample)
rng    = np.random.default_rng(42)
sample = emb_matrix[rng.choice(len(emb_matrix), size=min(500, len(emb_matrix)), replace=False)]
sim_matrix = sample @ sample.T
np.fill_diagonal(sim_matrix, 0)
avg_sim = float(sim_matrix.mean())
check("Average pairwise cosine similarity is reasonable (< 0.95)",
      avg_sim < 0.95,
      f"avg={avg_sim:.4f} — embeddings may have collapsed")
check("Average pairwise cosine similarity not too low (> 0.0)",
      avg_sim > 0.0,
      f"avg={avg_sim:.4f}")
print(f"         avg pairwise cosine (500-sample): {avg_sim:.4f}")

# ═══════════════════════════════════════════════════════════════
# 3. METADATA COMPLETENESS
# ═══════════════════════════════════════════════════════════════
section("3 · Metadata completeness")

REQUIRED_META = {
    "chunk_id","content_type","source","source_family",
    "edition_identifier","edition_name","language",
    "surah_number","ayah_number_in_surah","ayah_number_global","ayah_ref",
    "surah_name_arabic","juz","manzil","page","ruku","hizb_quarter","sajda",
    "raw_source_text","canonical_text","bm25_normalized_text",
    "embedding_normalized_text","canonical_text_sha256",
    "normalization_profile","chunk_schema_version","pipeline_version",
    "ingested_at_utc","metadata_fingerprint","source_url",
}
missing_meta_rows = [
    (i, sorted(REQUIRED_META - set(m)))
    for i, m in enumerate(all_metadatas)
    if REQUIRED_META - set(m)
]
check("All chunks have required metadata keys",
      not missing_meta_rows,
      f"{len(missing_meta_rows)} rows missing keys — first: {missing_meta_rows[:2]}")

# Per-edition composition
composition = Counter(
    (m["content_type"], m["language"], m["edition_identifier"])
    for m in all_metadatas
)
check("Edition composition matches expected counts",
      composition == Counter(EXPECTED_COUNTS),
      f"got {dict(composition)}")

# Canonical Quran coverage
canonical_refs = {
    m["ayah_ref"]
    for m in all_metadatas
    if m["edition_identifier"] == "quran-uthmani" and m["content_type"] == "quran_ayah"
}
check(f"Canonical Quran has {EXPECTED_QURAN_AYAH_TOTAL} unique ayah refs",
      len(canonical_refs) == EXPECTED_QURAN_AYAH_TOTAL,
      f"got {len(canonical_refs)}")
check(f"Canonical Quran has {EXPECTED_QURAN_SURAH_TOTAL} unique surahs",
      len({r.split(":")[0] for r in canonical_refs}) == EXPECTED_QURAN_SURAH_TOTAL)

for edition_id in SELECTED_EDITIONS:
    refs = {m["ayah_ref"] for m in all_metadatas if m["edition_identifier"] == edition_id}
    check(f"Edition {edition_id} ayah refs == canonical Quran refs",
          refs == canonical_refs,
          f"missing: {(canonical_refs - refs)!r:.60}  extra: {(refs - canonical_refs)!r:.60}")

# ═══════════════════════════════════════════════════════════════
# 4. TEXT & HASH INTEGRITY
# ═══════════════════════════════════════════════════════════════
section("4 · Text & hash integrity")

hash_failures, fingerprint_failures, id_failures, text_failures = [], [], [], []
for chunk_id, doc, meta in zip(all_ids, all_documents, all_metadatas):
    if chunk_id != meta["chunk_id"]:
        id_failures.append(chunk_id)
    if meta["raw_source_text"] != doc or meta["canonical_text"] != doc:
        text_failures.append(chunk_id)
    if hashlib.sha256(doc.encode()).hexdigest() != meta["canonical_text_sha256"]:
        hash_failures.append(chunk_id)
    expected_fp = QuranChunkBuilder._build_metadata_fingerprint(
        meta["edition_identifier"], meta["ayah_ref"],
        meta["canonical_text_sha256"], meta["source_url"],
    )
    if meta["metadata_fingerprint"] != expected_fp:
        fingerprint_failures.append(chunk_id)

check("chunk_id field matches Chroma ID for all chunks",
      not id_failures, f"{len(id_failures)} mismatches — first: {id_failures[:3]}")
check("raw_source_text == canonical_text == document for all chunks",
      not text_failures, f"{len(text_failures)} mismatches — first: {text_failures[:3]}")
check("SHA-256 hash correct for all chunks",
      not hash_failures, f"{len(hash_failures)} mismatches — first: {hash_failures[:3]}")
check("Metadata fingerprint correct for all chunks",
      not fingerprint_failures, f"{len(fingerprint_failures)} mismatches")

canonical_id_failures = [
    chunk_id for chunk_id, meta in zip(all_ids, all_metadatas)
    if meta["content_type"] == "quran_ayah" and meta["edition_identifier"] == "quran-uthmani"
    and chunk_id != f"quran_ar_{meta['surah_number']}_{meta['ayah_number_in_surah']}"
]
check("Canonical Quran chunk IDs follow quran_ar_S_A pattern",
      not canonical_id_failures,
      f"{len(canonical_id_failures)} wrong — first: {canonical_id_failures[:3]}")

# ═══════════════════════════════════════════════════════════════
# 5. EMBEDDING SIMILARITY — TASHKEEL INVARIANCE
# ═══════════════════════════════════════════════════════════════
section("5 · Embedding similarity — tashkeel / alef invariance")

SIMILARITY_THRESHOLD = 0.85
variant_pairs = [
    ("الرَّحْمَٰنِ",                            "الرحمن",                  "Ar-Rahman: tashkeel vs bare"),
    ("بِسْمِ اللَّهِ الرَّحْمَٰنِ الرَّحِيمِ",   "بسم الله الرحمن الرحيم",  "Basmala: tashkeel vs bare"),
    ("قُلْ هُوَ اللَّهُ أَحَدٌ",                 "قل هو الله احد",           "Al-Ikhlas 1: tashkeel vs bare"),
    ("إِنَّا أَعْطَيْنَاكَ الْكَوْثَرَ",          "انا اعطيناك الكوثر",       "Al-Kawthar 1: tashkeel vs bare"),
    ("اللَّهُ لَا إِلَٰهَ إِلَّا هُوَ الْحَيُّ الْقَيُّومُ",
                                               "الله لا اله الا هو الحي القيوم",
                                                                           "Ayat al-Kursi: tashkeel vs bare"),
    ("أولئك",                                  "اولئك",                    "Hamza-alef fold"),
    ("الرَّحِيمِ",                              "الرحيم",                   "Ar-Raheem: tashkeel vs bare"),
    ("\ufeffقل هو الله احد",                    "قل هو الله احد",           "BOM noise stripped"),
    ("قُلْ هُوَ اللَّهُ أَحَدٌ",                 "قل هو الله أحد",           "Partial diacritic removal"),
]

for left, right, label in variant_pairs:
    vecs = embedding_model.encode_queries([left, right])
    sim  = float(np.dot(vecs[0], vecs[1]))
    check(f"{label}  (sim={sim:.4f} ≥ {SIMILARITY_THRESHOLD})",
          sim >= SIMILARITY_THRESHOLD,
          f"left='{left}'  right='{right}'  sim={sim:.4f}")

# ═══════════════════════════════════════════════════════════════
# 6. EMBEDDING SEPARABILITY — DISTINCT AYAHS SHOULD BE DISSIMILAR
# ═══════════════════════════════════════════════════════════════
section("6 · Embedding separability — distinct ayahs dissimilar")

DISSIMILARITY_THRESHOLD = 0.98   # they should NOT be this close unless truly identical
contrastive_pairs = [
    ("قل هو الله احد",               "الحمد لله رب العالمين",  "Al-Ikhlas vs Al-Fatiha 2"),
    ("والعصر",                       "قل اعوذ برب الناس",      "Al-Asr 1 vs Al-Nas 1"),
    ("الله لا اله الا هو الحي القيوم","قل يا ايها الكافرون",   "Ayat al-Kursi vs Al-Kafirun 1"),
]
for left, right, label in contrastive_pairs:
    vecs = embedding_model.encode_queries([left, right])
    sim  = float(np.dot(vecs[0], vecs[1]))
    check(f"{label}  (sim={sim:.4f} < {DISSIMILARITY_THRESHOLD})",
          sim < DISSIMILARITY_THRESHOLD,
          f"embeddings suspiciously close: sim={sim:.4f}")

# ═══════════════════════════════════════════════════════════════
# 7. DENSE RETRIEVAL RECALL — KNOWN AYAHS MUST SURFACE
# ═══════════════════════════════════════════════════════════════
section("7 · Dense retrieval recall")

dense_probes = [
    ("قل هو الله احد",                "112:1", "quran_ayah", "quran-uthmani", "Al-Ikhlas 1"),
    ("الحمد لله رب العالمين",         "1:2",   "quran_ayah", "quran-uthmani", "Al-Fatiha 2"),
    ("الله لا اله الا هو الحي القيوم","2:255", "quran_ayah", "quran-uthmani", "Ayat al-Kursi"),
    ("والعصر ان الانسان لفي خسر",     "103:2", "quran_ayah", "quran-uthmani", "Al-Asr 2"),
    ("قل اعوذ برب الفلق",             "113:1", "quran_ayah", "quran-uthmani", "Al-Falaq 1"),
    ("انا اعطيناك الكوثر",            "108:1", "quran_ayah", "quran-uthmani", "Al-Kawthar 1"),
    ("قل هو الله احد",                "112:1", "tafsir",     "ar.muyassar",   "Al-Ikhlas tafsir muyassar"),
    ("الحمد لله رب العالمين",         "1:2",   "tafsir",     "ar.tabari",     "Al-Fatiha tafsir tabari"),
]

for query, expected_ref, ct, edition, label in dense_probes:
    results = retriever.dense_search(
        query, top_k=10, content_type=ct, edition_identifier=edition,
    )
    refs = [r["ref"] for r in results]
    check(f"Dense: {label}  →  {expected_ref} in top-10",
          expected_ref in refs,
          f"got {refs[:5]}")

# ═══════════════════════════════════════════════════════════════
# 8. EXACT REFERENCE LOOKUP
# ═══════════════════════════════════════════════════════════════
section("8 · Exact ayah-reference lookup")

ref_probes = [
    ("2:255",  "2:255",  "ASCII digits  2:255"),
    ("٢:٢٥٥", "2:255",  "Arabic-Indic  ٢:٢٥٥"),
    ("1:1",    "1:1",    "Al-Fatiha 1"),
    ("114:6",  "114:6",  "Al-Nas 6 (last ayah)"),
    ("2:286",  "2:286",  "Al-Baqara 286 (longest surah last)"),
    ("112:4",  "112:4",  "Al-Ikhlas 4"),
]

for query, expected_ref, label in ref_probes:
    results = retriever.exact_ayah_lookup(query)
    check(f"{label}",
          bool(results) and results[0]["ref"] == expected_ref,
          f"got {[r['ref'] for r in results][:3]}")

# ═══════════════════════════════════════════════════════════════
# 9. HYBRID SEARCH END-TO-END
# ═══════════════════════════════════════════════════════════════
section("9 · Hybrid search end-to-end")

hybrid_probes = [
    ("قل هو الله احد",                 "112:1", "quran_ayah", "quran-uthmani", None,  None, "Al-Ikhlas 1 bare"),
    ("\ufeffقل هو الله احد",            "112:1", "quran_ayah", "quran-uthmani", None,  None, "Al-Ikhlas BOM noise"),
    ("الله لا اله الا هو الحي القيوم", "2:255", "quran_ayah", "quran-uthmani", None,  None, "Ayat al-Kursi"),
    ("بسم الله الرحمن الرحيم",         "1:1",   "quran_ayah", "quran-uthmani", 1,     None, "Basmala surah_filter=1"),
    ("الحمد لله",                       "1:2",   "quran_ayah", "quran-uthmani", None,  1,    "Al-Fatiha 2 juz_filter=1"),
    ("قل هو الله احد",                 "112:1", "tafsir",     "ar.mukhtasar",  None,  None, "Al-Ikhlas tafsir mukhtasar"),
    ("قل هو الله احد",                 "112:1", "tafsir",     "ar.tabari",     None,  None, "Al-Ikhlas tafsir tabari"),
]

for query, expected_ref, ct, edition, surah, juz, label in hybrid_probes:
    results = retriever.hybrid_search(
        query, top_k=10,
        content_type=ct, edition_identifier=edition,
        surah_number=surah, juz=juz,
    )
    refs = [r["ref"] for r in results]
    check(f"Hybrid: {label}  →  {expected_ref} in top-10",
          expected_ref in refs,
          f"got {refs[:5]}")

# ═══════════════════════════════════════════════════════════════
# 10. CONTENT-TYPE & EDITION FILTERING ISOLATION
# ═══════════════════════════════════════════════════════════════
section("10 · Filtering isolation")

tafsir_results = retriever.hybrid_search("الرحمن الرحيم", top_k=10, content_type="tafsir")
check("Tafsir-only filter: all results are tafsir",
      bool(tafsir_results) and all(r["meta"]["content_type"] == "tafsir" for r in tafsir_results))

quran_results = retriever.hybrid_search(
    "الحمد لله رب العالمين", top_k=10, content_type="quran_ayah", edition_identifier="quran-uthmani",
)
check("Quran-only filter: all results are quran_ayah",
      bool(quran_results) and all(r["meta"]["content_type"] == "quran_ayah" for r in quran_results))
check("Quran-only filter: all results are quran-uthmani",
      all(r["meta"]["edition_identifier"] == "quran-uthmani" for r in quran_results))

for edition_id in SELECTED_EDITIONS:
    ed_results = retriever.hybrid_search(
        "قل هو الله احد", top_k=5,
        content_type="tafsir" if "." in edition_id else "quran_ayah",
        edition_identifier=edition_id,
    )
    check(f"Edition filter: all results are {edition_id}",
          not ed_results or all(r["meta"]["edition_identifier"] == edition_id for r in ed_results))

fatiha_results = retriever.hybrid_search(
    "بسم الله", top_k=10, content_type="quran_ayah",
    edition_identifier="quran-uthmani", surah_number=1,
)
check("Surah filter: all results are surah 1",
      not fatiha_results or all(r["meta"]["surah_number"] == 1 for r in fatiha_results))

juz1_results = retriever.hybrid_search(
    "الحمد لله", top_k=10, content_type="quran_ayah",
    edition_identifier="quran-uthmani", juz=1,
)
check("Juz filter: all results are juz 1",
      not juz1_results or all(r["meta"]["juz"] == 1 for r in juz1_results))

# ═══════════════════════════════════════════════════════════════
# 11. NORMALIZER SAFETY — RISKY FOLDS DISABLED
# ═══════════════════════════════════════════════════════════════
section("11 · Normalizer safety — risky folds disabled")

doc_tokens = normalizer.tokenize_document_for_bm25("رحمة رحمه على هدى هدي")
check("ة ↔ ه NOT collapsed in BM25 doc tokens",
      "رحمة" in doc_tokens and "رحمه" in doc_tokens)
check("ى ↔ ي NOT collapsed in BM25 doc tokens",
      "هدى"  in doc_tokens and "هدي"  in doc_tokens)

bare = normalizer.normalize_for_bm25_document("رَحْمَةً رَّحِيمًا")
check("Diacritics stripped in BM25 doc normalization",
      "َ" not in bare and "ً" not in bare)

query_folded = normalizer.normalize_for_bm25_query("أحمد إبراهيم آدم")
check("Alef-family folded in BM25 query normalization",
      "أ" not in query_folded and "إ" not in query_folded and "آ" not in query_folded,
      f"got: {query_folded}")

doc_unfolded = normalizer.normalize_for_bm25_document("أحمد إبراهيم آدم")
check("Alef-family NOT folded in BM25 doc normalization (conservative)",
      "أ" in doc_unfolded or "إ" in doc_unfolded,
      f"got: {doc_unfolded}")

ref_canonical = normalizer.normalize_ayah_ref("٢:٢٥٥")
check("Arabic-Indic digit normalisation in ayah ref",
      ref_canonical == "2:255", f"got {ref_canonical!r}")
ref_spaced = normalizer.normalize_ayah_ref("2 : 255")
check("Spaced ayah ref parsed correctly",
      ref_spaced == "2:255", f"got {ref_spaced!r}")

# ═══════════════════════════════════════════════════════════════
# 12. EMBEDDING COVERAGE — SPOT-CHECK STORED VECTORS
# ═══════════════════════════════════════════════════════════════
section("12 · Embedding coverage — stored vs freshly encoded")

spot_checks = ["1:1", "2:255", "112:1", "114:6"]
for ref in spot_checks:
    row = collection.get(
        where={"$and": [{"ayah_ref":            {"$eq": ref}},
                        {"edition_identifier":  {"$eq": "quran-uthmani"}}]},
        include=["documents", "metadatas", "embeddings"],
    )
    if not row["ids"]:
        check(f"Spot-check ayah {ref} exists in collection", False, "not found")
        continue

    stored_vec = np.array(row["embeddings"][0], dtype=np.float32)
    meta       = row["metadatas"][0]

    # ── Reconstruct the EXACT text_for_embedding used at ingest time ──────
    # QuranChunkBuilder builds:
    #   text_for_embedding = passage_prefix + "سورة {name} آية {n}: " + embedding_norm
    surah_name    = meta["surah_name_arabic"]
    ayah_num      = meta["ayah_number_in_surah"]
    embedding_norm = meta["embedding_normalized_text"]
    passage_prefix = config.embedding_passage_prefix          # "passage: "
    ingest_prefix  = f"سورة {surah_name} آية {ayah_num}: "
    text_for_embedding = passage_prefix + ingest_prefix + embedding_norm

    fresh_vec = embedding_model.encode_passages([text_for_embedding])[0]

    s_norm = stored_vec / np.linalg.norm(stored_vec)
    f_norm = fresh_vec  / np.linalg.norm(fresh_vec)
    sim    = float(np.dot(s_norm, f_norm))

    check(f"Stored vs fresh embedding for {ref}  (sim={sim:.4f} ≥ 0.97)",
          sim >= 0.97,
          f"possible embedding drift: sim={sim:.4f}\n"
          f"         text used: {text_for_embedding[:80]}…")

# ═══════════════════════════════════════════════════════════════
# SUMMARY
# ═══════════════════════════════════════════════════════════════
print(f"\n{'═'*60}")
if _failures:
    print(f"  {len(_failures)} CHECK(S) FAILED:")
    for f in _failures:
        print(f"    • {f}")
    raise AssertionError(f"{len(_failures)} validation check(s) failed — see above.")
else:
    total_checks = sum(1 for line in open("/dev/stdin") if PASS in line) # approximate
    print(f"  ALL CHECKS PASSED ✓")
    print(f"  Corpus: {stored_count:,} chunks across {len(SELECTED_EDITIONS)} editions")
    print(f"  Embedding model: {config.embedding_model_name}")
    print(f"  Collection: {config.chroma_collection_name}")
    print(f"  Zip artifact: {EXPORT_ZIP_PATH}")
print(f"{'═'*60}")


────────────────────────────────────────────────────────────
  1 · Collection integrity
────────────────────────────────────────────────────────────
  ✓ PASS  Total chunk count matches expected
  ✓ PASS  IDs count
  ✓ PASS  Docs count
  ✓ PASS  Metadata count
  ✓ PASS  Embeddings count
  ✓ PASS  No duplicate IDs
  ✓ PASS  No empty documents
  ✓ PASS  All language=ar
  ✓ PASS  Edition set matches config
  ✓ PASS  Source field only quran/tafsir

────────────────────────────────────────────────────────────
  2 · Embedding vector sanity
────────────────────────────────────────────────────────────
  ✓ PASS  Embedding matrix is 2-D
  ✓ PASS  Embedding dimension = 1024
  ✓ PASS  No NaN values in any embedding
  ✓ PASS  No all-zero embeddings
  ✓ PASS  All embeddings are unit-normalised (norm ∈ [0.99, 1.01])
  ✓ PASS  Average pairwise cosine similarity is reasonable (< 0.95)
  ✓ PASS  Average pairwise cosine similarity not too low (> 0.0)
         avg pairwise cosine (500-sample): 0.4573

───

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

  ✓ PASS  Ar-Rahman: tashkeel vs bare  (sim=0.9990 ≥ 0.85)


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

  ✓ PASS  Basmala: tashkeel vs bare  (sim=1.0063 ≥ 0.85)


...

  ✓ PASS  Stored vs fresh embedding for 2:255  (sim=1.0000 ≥ 0.97)


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

  ✓ PASS  Stored vs fresh embedding for 112:1  (sim=1.0000 ≥ 0.97)


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

  ✓ PASS  Stored vs fresh embedding for 114:6  (sim=1.0000 ≥ 0.97)

════════════════════════════════════════════════════════════
  ALL CHECKS PASSED ✓
  Corpus: 24,944 chunks across 4 editions
  Embedding model: jinaai/jina-embeddings-v3
  Collection: quran_tafsir_ar
  Zip artifact: /kaggle/working/quran_rag_build/quran_full.zip
════════════════════════════════════════════════════════════
